# Full VisDrone detector training (Colab)

Select a GPU runtime before running. This notebook uses the committed full-data config as-is. Dataset files, checkpoints, and results are stored on Google Drive. If Colab disconnects, reconnect, run the setup and path cells again, then rerun training; the notebook passes `--resume` automatically when `last.pt` exists.

The configured run uses all 6,471 training images for 10 epochs and evaluates all 548 validation images after each epoch. This is a multi-hour run and may require more than one Colab session; exact full-run time has not been measured. VisDrone is treated as non-commercial and share-alike.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount("/content/drive")

REPO_URL = "https://github.com/elgunismayil0v2001/vision-engine.git"
REPO_DIR = Path("/content/vision-engine")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Repository:", REPO_DIR)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. Select a GPU runtime and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))

DRIVE_ROOT = Path("/content/drive/MyDrive/vision-engine-visdrone")
DATA_ROOT = DRIVE_ROOT / "data" / "visdrone2019-det"
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "visdrone2019_detector"
RESULTS_DIR = DRIVE_ROOT / "results" / "visdrone"
CONFIG = "train/configs/visdrone.yaml"
RESUME_CHECKPOINT = OUTPUT_DIR / "last.pt"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_ROOT)
print("Checkpoints and metrics:", OUTPUT_DIR)
print("Final results:", RESULTS_DIR)

## Download the pinned full dataset

This command downloads the configured full train and validation splits into Drive. Rerunning it after reconnecting reuses files already present at the dataset path.

In [ ]:
download_command = [
    sys.executable, "-m", "train.download_visdrone",
    "--config", CONFIG,
    "--data-root", str(DATA_ROOT),
]
print("Running:", " ".join(download_command))
subprocess.run(download_command, cwd=REPO_DIR, check=True)

## Train or resume

On the first run, training starts from the configured COCO weights. After a disconnect, the same cell detects the Drive checkpoint and adds `--resume <output-dir>/last.pt`. Training settings remain in `train/configs/visdrone.yaml`.

In [ ]:
train_command = [
    sys.executable, "-m", "train.detector",
    "--config", CONFIG,
    "--device", "auto",
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(OUTPUT_DIR),
]
if RESUME_CHECKPOINT.is_file():
    train_command.extend(["--resume", str(RESUME_CHECKPOINT)])
    print("Resuming from:", RESUME_CHECKPOINT)
else:
    print("Starting a new run from the configured COCO weights.")
print("Running:", " ".join(train_command))
subprocess.run(train_command, cwd=REPO_DIR, check=True)

In [ ]:
import shutil

summary_path = OUTPUT_DIR / "summary.json"
metrics_path = OUTPUT_DIR / "metrics.jsonl"
run_path = OUTPUT_DIR / "run.json"
for path in (summary_path, metrics_path, run_path):
    if not path.is_file():
        raise FileNotFoundError(f"Training output is missing: {path}")

shutil.copy2(summary_path, RESULTS_DIR / "raw.json")
shutil.copy2(metrics_path, RESULTS_DIR / "metrics.jsonl")
shutil.copy2(run_path, RESULTS_DIR / "run.json")
print("Copied raw results and metrics to:", RESULTS_DIR)